# Dim Estabelecimento

In [0]:
import pyspark.sql.functions as f

In [0]:
df_tru = spark.read.table('saude_sus.trusted.tru_estabelecimento')

In [0]:
df_dim = (
    df_tru
    .select(
        f.md5(f.concat_ws("-", f.trim(f.upper(f.col("COD_CNES"))), f.col("DAT_ATUALIZACAO"))).alias("SK_ESTABELECIMENTO"),
        f.col("COD_CNES").alias("COD_CNES"),
        f.col("NOM_FANTASIA").alias("NOM_FANTASIA"),
        f.col("NOM_RAZAO_SOCIAL").alias("NOM_RAZAO_SOCIAL"),
        f.col("DSC_ESFERA_ADMINISTRATIVA").alias("DSC_ESFERA_ADMINISTRATIVA"),
        f.col("DAT_ATUALIZACAO").alias("DAT_ATUALIZACAO"),
        f.col("DAT_INICIO_VIGENCIA"),
        f.col("DAT_FIM_VIGENCIA"),
        f.col("FLG_VIGENTE")
    )      
)

In [0]:
try:
    df_dim_existente = spark.table("saude_sus.refined.dim_estabelecimento").select("SK_ESTABELECIMENTO")
except:
    df_dim_existente = spark.createDataFrame([], df_dim.select("SK_ESTABELECIMENTO").schema)

df_dim_limpo = df_dim.distinct()

df_para_inserir = df_dim_limpo.join(
    df_dim_existente, 
    on="SK_ESTABELECIMENTO", 
    how="left_anti"
)
print(f"Dados populados na dim estabelecimento: {df_para_inserir.count()}")

df_para_inserir.write.format("delta").mode("append").saveAsTable("saude_sus.refined.dim_estabelecimento")
spark.sql('OPTIMIZE saude_sus.refined.dim_estabelecimento ZORDER BY (COD_CNES)')